# Evaluate note-level diagnosis

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_diagnosis_label
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Read in true labels


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
true_training = pd.read_excel(readfiletrue)

In [ ]:
true_training.head()

In [ ]:
len(true_training)

In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
true_testing = pd.read_excel(readfiletrue)

In [ ]:
true_testing.head()

In [ ]:
len(true_testing)

In [ ]:
train_ids = true_training["PatientSSN"].unique()
test_ids = true_testing["PatientSSN"].unique()

In [ ]:
len(train_ids)

In [ ]:
len(test_ids)

## Read in already generated results


In [ ]:
readfileresults = paths.data("Random500/Results/testing_fewshot_binary.xlsx")
raw_predictions = pd.read_excel(readfileresults)

In [ ]:
results_s1 = raw_predictions[
    ["PatientSSN", "NoteDate", "NoteText", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]
]

In [ ]:
results_s1.head()

In [ ]:
readfileresults = paths.data("Second237/Results/phase1_fewshot_binary.xlsx")
raw_predictions = pd.read_excel(readfileresults)

In [ ]:
results_s2 = raw_predictions[
    ["PatientSSN", "NoteDate", "NoteText", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]
]

In [ ]:
results_s2.head()

In [ ]:
combined_results = pd.concat([results_s1, results_s2], ignore_index=True)

In [ ]:
results_all = combined_results[combined_results["PatientSSN"].isin(test_ids)]

In [ ]:
len(results_all)

In [ ]:
results_all.head()

## Combine true and result data for performance


In [ ]:
final = true_testing.merge(
    results_all[["PatientSSN", "Note_index", "SMMDx_LLM_Binary"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)

## POST PROCESSING


In [ ]:
llm_columns = ["SMMDx_LLM_Binary", "SMM Dx (final)"]
final = add_parsed_labels(final, llm_columns, parse_diagnosis_label)

In [ ]:
final.head(5)

## Note-level BINARY CLASSIFICATION PERFORMANCE


In [ ]:
conditions = [
    (final["post_SMM Dx (final)"] == "Yes") & (final["post_SMMDx_LLM_Binary"] == "Yes"),
    (final["post_SMM Dx (final)"] == "No") & (final["post_SMMDx_LLM_Binary"] == "No"),
    (final["post_SMM Dx (final)"] == "Yes") & (final["post_SMMDx_LLM_Binary"] == "No"),
    (final["post_SMM Dx (final)"] == "No") & (final["post_SMMDx_LLM_Binary"] == "Yes"),
]
choices = ["TP", "TN", "FN", "FP"]
final["Outcome1_Binary"] = np.select(conditions, choices, default="Unknown")

In [ ]:
final_error = final[final["Outcome1_Binary"].isin(["FP", "FN"])]

In [ ]:
final_error["NoteText"] = (
    final_error["NoteText"]
    .str.replace("_x000D_", "", regex=False)
    .str.replace("\r", "\n", regex=False)
)

In [ ]:
erroranalysis = paths.data("Update/Results/phase1errors_testing_fewshot.xlsx")
final_error.to_excel(erroranalysis, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
final["y_true"] = final["post_SMM Dx (final)"].str.lower().map({"yes": 1, "no": 0})
final["y_pred"] = final["post_SMMDx_LLM_Binary"].str.lower().map({"yes": 1, "no": 0})

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["Non-SMM", "SMM"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)